In [ ]:
import os, glob
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from tqdm import tqdm
from l0_data import RAW_ROOT, PROCESSED_ROOT, require_files, MSU_OBS_PATH


In [ ]:
obs_data_path = MSU_OBS_PATH

msu_time_belt = 3

df_msu = pd.read_csv (obs_data_path, sep = ';', parse_dates=['DATA'], dayfirst=True).rename(columns={'T33': 'rh', 'TB':'ta', 'PPP':'ps'})[['DATA', 'SR', 'ta', 'rh', 'ps']]

df_msu['DATA'] = df_msu['DATA'] + pd.Series ([pd.Timedelta(hours=x) for x in df_msu['SR']], index = df_msu.index) - pd.Timedelta (hours=msu_time_belt)

df_msu = df_msu.set_index ('DATA')
df_msu['rho'] = (df_msu['ps']*100) / (286.7 * (df_msu['ta'] + 273.15))
df_msu.head()

In [ ]:
import xarray as xr
gradient_obs_path = str(RAW_ROOT / "Moscow" / "MSU mast" / "Gradients")
grad_data4levels = {}
for level_name in ['M1', 'M2']:
    files = require_files(gradient_obs_path + '/' + level_name + '/*/*/*.nc')
    dfs = []
    for file in tqdm(files):
        with xr.open_dataset(file, engine='netcdf4') as dataset:
            dfs.append(dataset.to_dataframe())
    grad_data4levels [level_name] = pd.concat(dfs)

In [23]:
grad_data4levels_rs = {key:item.resample('30min').mean() for key, item in grad_data4levels.items()}

In [ ]:
twr_data_dir = str(RAW_ROOT / "Moscow" / "MSU mast" / "FLUXES_MSU")

sites = ['A1', 'A2', 'A3']

levels = {'A1': 2.2, 'A2': 11.1, 'A3': 18.8}

df4sites = {}
for site in sites:
    df4site = pd.DataFrame()
    files = require_files(twr_data_dir + '/' + site + '/*/*turb_stats*.csv')
    for file in tqdm (files):
        cur_df = pd.read_csv (file, sep = ',', parse_dates=[0], index_col=[0])
        df4site = pd.concat((df4site, cur_df))

    time_arr = pd.date_range(pd.Timestamp(df4site.index.date[0]), pd.Timestamp(df4site.index.date[-1]), freq = '30min') 
    df4site = df4site.resample('30min').last() 
    df4site = df4site.reindex(time_arr)

    df_msu_sel = df_msu[df4site.index[0]:df4site.index[-1]].resample('30min').last().interpolate(method='linear')
    df4site = df4site.merge(df_msu_sel.rename(columns = {c:c + '_obs' for c in df_msu_sel.columns}), how = 'left', left_index = True, right_index = True)
    for key, item in grad_data4levels_rs.items():
        df4site = df4site.merge(item.rename(columns = {c:c + '_' + key for c in item.columns}), how = 'left', left_index = True, right_index = True)
        
    df4site['H'] = df4site['wt'] * df4site['rho_obs'] * 1003
    df4site['tau'] = df4site['u_star']**2 * df4site['rho_obs'] 
    df4site['wind_vel'] = np.abs(df4site['u_rot'])
    df4site['sigma_U'] = np.sqrt(df4site['uu'])
    df4site['sigma_V'] = np.sqrt(df4site['vv'])
    df4site['sigma_W'] = np.sqrt(df4site['ww'])

    df4sites[str(int (np.round (levels[site]))) + 'm'] = df4site

    #display(files)

In [50]:
pd.options.plotting.backend = 'plotly'
df4sites['19m'][['t', 'ta_obs', 'temp_M1', 'temp_M2']].plot()

In [4]:
# %matplotlib qt
# df4sites['A1']['tau'].resample('D').mean().plot()
# df4sites['A2']['tau'].resample('D').mean().plot()
# df4sites['A3']['tau'].resample('D').mean().plot()

In [5]:
# df4sites['A1']['2022':'2023']['t'].plot()
# df4sites['A2']['2022':'2023']['t'].plot()
# df4sites['A3']['2022':'2023']['t'].plot()
# df_msu['2022':'2023']['ta'].plot()

In [6]:
(df4site['sigma_W'] / df4site['u_star']).median()

1.2985616543893164

In [ ]:
export_dir = str(PROCESSED_ROOT / "Moscow")
os.makedirs(export_dir, exist_ok=True)

for level in df4sites.keys():
    df4site = df4sites[level]
    df2export = df4site[['H', 'tau', 'u_star', 'ta_obs', 'rh_obs', 'ps_obs', 'rho_obs', 'u', 'v', 'w', 'sigma_U', 'sigma_V', 'sigma_W', 'wind_vel']]
    df2export = df2export.rename(columns = {'H': 'Qh',
                                            'tau': 'Qtau',
                                            'u_star':'Ustar', 
                                            'ta_obs': 'T_air',
                                            'rh_obs': 'RH_air',
                                            'ps_obs': 'P_air',
                                            'rho_obs': 'Rho_air',
                                            'v':'Wind_V',
                                            'u':'Wind_U',
                                            'w':'Wind_W',
                                            'wind_vel':'Wind_vel',
                                            'sigma_U':'Wind_U_std',
                                            'sigma_V':'Wind_V_std', 
                                            'sigma_W':'Wind_W_std'})
    df2export.index = df2export.index
    df2export.to_csv(export_dir + '/' + 'MSU_' +level +  '.csv')
